In [ ]:
sub_rfm, sub_churn, sub_forecast, sub_clv = st.tabs(
    ["Customer Segments", "Churn / Win-back Risk", "Demand Forecast", "Customer Lifetime Value"]
)

In [ ]:
with sub_rfm:
    result = aa.compute_customer_segments(data)
    if not result:
        st.info("Not enough customer/order data to compute customer segments.")
    else:
        seg_summary = result["segment_summary"]
        seg_order = seg_summary.index.tolist()

        c1, c2 = st.columns(2)
        with c1:
            seg_counts = seg_summary.reset_index().sort_values("Customers", ascending=False)
            fig = px.bar(seg_counts, x="Customers", y="Segment", orientation="h",
                          title="Customers by Segment", color_discrete_sequence=[ACCENT1])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width="stretch")
        with c2:
            seg_rev = seg_summary.reset_index().sort_values("Total_Revenue", ascending=False)
            fig = px.bar(seg_rev, x="Total_Revenue", y="Segment", orientation="h",
                          title="Revenue by Segment", color_discrete_sequence=[ACCENT2])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width="stretch")

        st.subheader("Segment Summary")
        st.dataframe(seg_summary, width="stretch")

        st.caption(
            "Segments combine Recency (days since last order), raw Frequency (1 / 2 / 3+ orders), and "
            "Monetary (lifetime spend, scored 1-5). Because 94%+ of customers here are one-time "
            "buyers, segments split mainly on whether they've ever repeated (Champions / Loyal / "
            "At-Risk Repeat Buyers) vs. how recent and how big their one order was (Big Spenders, "
            "New, Promising, Fading, Lost). 'Avg Churn Risk %' is the model's raw return probability "
            "(low across the board since the brand's overall repeat rate is only ~2%), while "
            "'Avg Churn Risk Percentile' ranks each segment's churn risk *relative to all other "
            "customers* (0-100) -- use the percentile to compare urgency across segments, and the "
            "raw % only within a segment. Avg/Total Predicted CLV come from the CLV model below, so "
            "each segment shows both its win-back urgency and its value."
        )

        if result["has_churn"] and result["has_clv"]:
            st.subheader("Priority Matrix: Value vs Churn Risk")
            quad = seg_summary.reset_index().dropna(subset=["Avg_Predicted_CLV", "Avg_Churn_Risk_Percentile"])
            if not quad.empty:
                fig = px.scatter(
                    quad, x="Avg_Predicted_CLV", y="Avg_Churn_Risk_Percentile", size="Customers",
                    color="Segment", hover_name="Segment", text="Segment",
                    title="Segment Priority: higher-right = high value, high relative churn risk = act now",
                )
                fig.update_traces(textposition="top center")
                fig.update_layout(xaxis_title="Avg Predicted CLV (₹)", yaxis_title="Avg Churn Risk Percentile (vs. all customers)")
                st.plotly_chart(fig, width="stretch")

            watchlist = result["watchlist"]
            if not watchlist.empty:
                st.subheader("Priority Watchlist")
                st.caption(
                    "High-value, repeat-prone customers (Champions / At-Risk Champions / Loyal "
                    "Customers / Big Spenders) the churn model flags as High Risk -- ranked by "
                    "Predicted CLV. These are the highest-stakes win-back targets."
                )
                cols = ["Customer Key", "Segment", "Recency", "Frequency", "Monetary",
                        "Churn_Risk_Pct", "Churn_Risk_Percentile", "Predicted_CLV", "Recommended Action"]
                st.dataframe(watchlist[cols].round(2), width="stretch")
        elif not result["has_churn"]:
            st.info("Not enough historical paid-order data to layer in churn risk yet -- showing RFM + CLV only where available.")
        elif not result["has_clv"]:
            st.info("Not enough customer data to layer in predicted CLV yet -- showing RFM + churn risk only.")

In [ ]:
with sub_churn:
    result = aa.compute_churn_model(data)
    if not result:
        st.info("Not enough historical paid-order data to train a churn/win-back model.")
    else:
        scored = result["scored_customers"]
        overall_rate = result["overall_return_rate"] * 100

        c1, c2, c3, c4 = st.columns(4)
        with c1:
            kpi_card("Model AUC", f"{result['auc']:.2f}" if result["auc"] is not None else "n/a")
        with c2:
            kpi_card("Prediction Window", f"{result['window_days']} days")
        with c3:
            kpi_card("Baseline Reorder Rate", f"{overall_rate:.2f}%")
        with c4:
            high_risk_pct = (scored["Risk_Tier"] == "High Risk").mean() * 100
            kpi_card("High-Risk Customers", f"{high_risk_pct:.1f}%", positive=False)

        st.caption(
            f"Predicts the probability each customer places another paid order within the next "
            f"{result['window_days']} days, using their order history (recency, frequency, spend, "
            f"discount/COD/RTO behavior, location, and product mix: average item price, average pack "
            f"size, primary product line, and the calendar month of their last purchase). "
            f"'Churn Risk' = 1 - return probability. "
            f"Because the brand's overall repeat rate is low ({overall_rate:.1f}%), use this to "
            f"*rank and prioritize* win-back targeting rather than as an absolute forecast."
        )

        col1, col2 = st.columns(2)
        with col1:
            tier_counts = scored["Risk_Tier"].value_counts().reindex(["High Risk", "Medium Risk", "Low Risk"]).reset_index()
            tier_counts.columns = ["Risk Tier", "Customers"]
            fig = px.bar(tier_counts, x="Risk Tier", y="Customers", title="Customers by Churn Risk Tier",
                          color="Risk Tier",
                          color_discrete_map={"High Risk": DANGER, "Medium Risk": "#C9A14A", "Low Risk": SUCCESS})
            st.plotly_chart(fig, width="stretch")
        with col2:
            imp = result["feature_importance"].head(8).reset_index()
            imp.columns = ["Feature", "Importance"]
            fig = px.bar(imp, x="Importance", y="Feature", orientation="h",
                          title="What Drives the Prediction", color_discrete_sequence=[ACCENT1])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width="stretch")

        st.subheader("High-Value Customers at Risk")
        st.caption("High-spend customers with the highest predicted churn risk -- good win-back targets.")
        at_risk = scored[scored["Risk_Tier"] == "High Risk"].sort_values("Monetary", ascending=False).head(20)
        st.dataframe(
            at_risk[["Customer Key", "Recency", "Frequency", "Monetary", "AOV", "Churn_Risk_Pct"]].round(2),
            width="stretch",
        )

In [ ]:
with sub_forecast:
    horizon = st.slider("Forecast horizon (months)", 1, 3, 3, key="forecast_horizon")
    results = aa.compute_forecast(data, periods=3)
    if not results:
        st.info("Not enough line-item history to build a demand forecast.")
    else:
        summary_rows = []
        for cat, r in results.items():
            hist, fcst = r["history"], r["forecast"]
            fcst = fcst.iloc[:horizon]

            fig = go.Figure()
            fig.add_scatter(x=hist.index, y=hist.values, mode="lines+markers", name="Actual",
                             line=dict(color=ACCENT1))
            fig.add_scatter(x=fcst.index, y=fcst.values, mode="lines+markers", name="Forecast",
                             line=dict(color=ACCENT2, dash="dash"))
            fig.update_layout(title=f"{cat} - Monthly Units (History + Forecast)", yaxis_title="Units")
            st.plotly_chart(fig, width="stretch")

            for month, val in fcst.items():
                summary_rows.append({"Category": cat, "Month": month.strftime("%Y-%m"), "Forecasted Units": round(val, 1)})

        st.subheader("Forecast Summary")
        summary_df = pd.DataFrame(summary_rows)
        pivot = summary_df.pivot(index="Category", columns="Month", values="Forecasted Units")
        st.dataframe(pivot, width="stretch")

        st.caption(
            "Categories shown are the top sellers by revenue over the last 6 months (not all-time), "
            "so discontinued/limited-edition lines drop off once they stop selling. Forecasts use "
            "Holt-Winters exponential smoothing with a damped trend (trend + yearly seasonality where "
            "24+ months of history are available and the category is still active, trend-only "
            "otherwise) on monthly units sold per category. The damped trend prevents forecasts from "
            "running away to zero or extrapolating an old seasonal spike indefinitely; categories "
            "whose recent sales have fallen below 15% of their historical peak skip the seasonal "
            "component entirely to avoid false 'comeback' spikes."
        )

In [ ]:
with sub_clv:
    result = aa.compute_clv(data)
    if not result:
        st.info("Not enough customer data to estimate lifetime value.")
    else:
        customers_clv = result["customers"]

        c1, c2, c3 = st.columns(3)
        with c1:
            kpi_card("Avg Predicted CLV", f"₹{customers_clv['Predicted_CLV'].mean():,.0f}")
        with c2:
            kpi_card("Repeat Customer Rate", f"{result['repeat_rate']*100:.2f}%")
        with c3:
            kpi_card("Avg Customer Lifespan", f"{result['avg_lifespan_days']:.0f} days")

        st.caption(
            "Predicted CLV = Avg Order Value x Annual Purchase Frequency x Avg Customer Lifespan, "
            "computed from repeat-customer behavior. One-time buyers get an expected-value CLV based "
            "on the brand's overall repeat rate. Treat as directional, not exact."
        )

        col1, col2 = st.columns(2)
        with col1:
            if not result["by_location"].empty:
                loc = result["by_location"].reset_index()
                fig = px.bar(loc, x="Total_Predicted_CLV", y="Shipping Province", orientation="h",
                              title="Total Predicted CLV by State (Top 15)", color_discrete_sequence=[ACCENT1])
                fig.update_layout(yaxis=dict(autorange="reversed"))
                st.plotly_chart(fig, width="stretch")
        with col2:
            if not result["by_category"].empty:
                cat = result["by_category"].reset_index()
                fig = px.bar(cat, x="Total_Predicted_CLV", y="Primary Category", orientation="h",
                              title="Total Predicted CLV by Primary Product Category (Top 15)",
                              color_discrete_sequence=[ACCENT2])
                fig.update_layout(yaxis=dict(autorange="reversed"))
                st.plotly_chart(fig, width="stretch")

        st.subheader("CLV by State")
        st.dataframe(result["by_location"], width="stretch")

        st.subheader("CLV by Primary Product Category")
        st.dataframe(result["by_category"], width="stretch")